# 🏠 Grupo 4 — Ames Housing | Fase 2
## Notebook 02: Data Preparation — Pipeline de Preprocesamiento
**CRISP-DM:** Data Preparation

---
### ¿Qué hacemos aquí?
Construimos el **pipeline reproducible** de preprocesamiento:
1. Eliminación de outliers (documentados por De Cock, 2011)
2. Mapeo ordinal de variables categóricas con orden semántico
3. Feature engineering (variables derivadas)
4. Partición estratificada 70/15/15
5. ColumnTransformer sklearn (ajustado SOLO en train)
6. Verificación anti-leakage

**Principio CRISP-DM:** El pipeline se ajusta solo en train y se aplica en val/test. Así garantizamos que no hay fuga de información.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_ames_housing, load_config
from src.preprocessing import (
    remove_outliers, apply_ordinal_mappings, engineer_features,
    stratified_split, build_preprocessor, get_feature_groups, prepare_data
)
from src.utils import set_plot_style, usd_formatter, save_fig
import matplotlib.ticker as mticker

set_plot_style()
cfg = load_config('config/params.yaml')
print('✅ Config cargada')
print(f'   random_state = {cfg["random_state"]}')
print(f'   log_transform_target = {cfg["preprocessing"]["log_transform_target"]}')

In [ ]:
# ── Cargar dataset crudo ─────────────────────────────────────────────────────
df_raw = load_ames_housing('data/raw/ames_housing_raw.csv')
print(f'Dataset crudo: {df_raw.shape}')

## 1. Eliminación de Outliers

In [ ]:
# ── Eliminar propiedades atípicas (documentadas por De Cock, 2011) ───────────
# Son ventas parciales de terreno o propiedades no-residenciales:
# GrLivArea > 4000 ft² CON precio bajo (<$300k)

df_clean = remove_outliers(df_raw, cfg)
print(f'Registros restantes: {len(df_clean):,}')

## 2. Mapeo de Variables Ordinales

In [ ]:
# ── Convertir variables ordinales a numéricas ────────────────────────────────
# Ejemplos: Exter_Qual: Po(1) < Fa(2) < TA(3) < Gd(4) < Ex(5)
#           Bsmt_Qual: No_Basement(0) < Po(1) < ... < Ex(5)
# Esto preserva el orden semántico y es mucho mejor que OHE para variables ordinales.

df_ord = apply_ordinal_mappings(df_clean)

# Verificar que las conversiones quedaron bien
print('Distribución Overall_Qual tras conversión:')
print(df_ord['Overall_Qual'].value_counts().sort_index())
print(f'\nDtypes que cambiaron a numérico:')
changed = [c for c in df_ord.columns
           if df_ord[c].dtype != df_clean[c].dtype
           and pd.api.types.is_numeric_dtype(df_ord[c])]
print(f'  {len(changed)} columnas convertidas a ordinal numérico')
print(f'  Ejemplos: {changed[:5]}')

## 3. Feature Engineering

In [ ]:
# ── Crear variables derivadas ────────────────────────────────────────────────
# Estas features capturan combinaciones relevantes que los modelos
# no siempre aprenden solos de las variables individuales.

df_eng = engineer_features(df_ord)

new_features = ['Total_SF', 'Total_Baths', 'House_Age', 'Remod_Age',
                'Was_Remodeled', 'Total_Porch_SF', 'Has_Pool']

print('Nuevas features creadas:')
for feat in new_features:
    if feat in df_eng.columns:
        print(f'  {feat:20s}: {df_eng[feat].describe()["mean"]:,.1f} (media)')

# Correlación de las nuevas features con Sale_Price
print('\nCorrelación de features engineered con Sale_Price:')
for feat in new_features:
    if feat in df_eng.columns and pd.api.types.is_numeric_dtype(df_eng[feat]):
        c = df_eng[feat].corr(df_eng['Sale_Price'])
        print(f'  {feat:20s}: r = {c:.3f}')

## 4. Partición Estratificada 70/15/15

In [ ]:
# ── Partición estratificada por bins de precio ───────────────────────────────
# Estratificamos para asegurar que los tres splits tienen la misma
# distribución de precios. Evita que train tenga solo casas baratas, etc.

train_df, val_df, test_df = stratified_split(df_eng, cfg)

print(f'\n  Train : {len(train_df):,} ({len(train_df)/len(df_eng)*100:.1f}%)')
print(f'  Val   : {len(val_df):,} ({len(val_df)/len(df_eng)*100:.1f}%)')
print(f'  Test  : {len(test_df):,} ({len(test_df)/len(df_eng)*100:.1f}%)')

# Verificar que las distribuciones de precio son similares en los 3 splits
fig, ax = plt.subplots(figsize=(10, 5))
bins_hist = np.linspace(df_eng['Sale_Price'].min(), df_eng['Sale_Price'].max(), 40)
for split, label, color in [(train_df, 'Train (70%)', '#1F3864'),
                             (val_df,   'Val (15%)',   '#2E74B5'),
                             (test_df,  'Test (15%)',  '#BDD7EE')]:
    ax.hist(split['Sale_Price'], bins=bins_hist, density=True,
            alpha=0.5, label=label, color=color)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(usd_formatter))
ax.set_title('Distribución de Sale_Price por Split\n(estratificado → distribuciones similares ✅)', fontweight='bold')
ax.set_xlabel('Sale_Price')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
save_fig('11_split_distribution')
plt.show()

print('\n✅ Verificación: medianas de precio por split')
for split, name in [(train_df,'Train'), (val_df,'Val'), (test_df,'Test')]:
    print(f'   {name}: ${split["Sale_Price"].median():,.0f}')

## 5. Pipeline sklearn — ColumnTransformer

In [ ]:
# ── Construir y ajustar el preprocessor ─────────────────────────────────────
# REGLA CRÍTICA: fit() SOLO en train. transform() en val y test.
# Si ajustamos en todo el dataset, el scaler 'vería' los datos de test
# → data leakage → métricas infladas artificialmente.

target_col = cfg['dataset']['target_col']

# Separar X / y
X_train_df = train_df.drop(columns=[target_col])
X_val_df   = val_df.drop(columns=[target_col])
X_test_df  = test_df.drop(columns=[target_col])

y_train_raw = train_df[target_col].values
y_val_raw   = val_df[target_col].values
y_test_raw  = test_df[target_col].values

# Log-transform del target
y_train = np.log1p(y_train_raw)
y_val   = np.log1p(y_val_raw)
y_test  = np.log1p(y_test_raw)

# Identificar columnas por tipo
num_cols, cat_cols = get_feature_groups(X_train_df)
print(f'Columnas numéricas  : {len(num_cols)}')
print(f'Columnas categóricas: {len(cat_cols)}')

# Construir preprocessor
preprocessor = build_preprocessor(num_cols, cat_cols)

# FIT en train, TRANSFORM en todos
X_train = preprocessor.fit_transform(X_train_df)   # ← fit + transform
X_val   = preprocessor.transform(X_val_df)          # ← solo transform
X_test  = preprocessor.transform(X_test_df)         # ← solo transform

# Recuperar nombres de features
cat_feature_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + cat_feature_names

print(f'\nFeatures tras OHE: {len(feature_names)}')
print(f'X_train shape    : {X_train.shape}')
print(f'X_val   shape    : {X_val.shape}')
print(f'X_test  shape    : {X_test.shape}')
print('\n✅ Pipeline ajustado SOLO en train. No hay data leakage.')

In [ ]:
# ── Guardar artefactos para los notebooks siguientes ────────────────────────
import joblib, json

os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Guardar preprocessor ajustado
joblib.dump(preprocessor, 'models/preprocessor.pkl')

# Guardar splits como CSV
train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)

# Guardar arrays numpy (más rápido para cargar en siguientes notebooks)
np.save('data/processed/X_train.npy', X_train)
np.save('data/processed/X_val.npy',   X_val)
np.save('data/processed/X_test.npy',  X_test)
np.save('data/processed/y_train.npy', y_train)
np.save('data/processed/y_val.npy',   y_val)
np.save('data/processed/y_test.npy',  y_test)

# Guardar nombres de features
with open('data/processed/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print('✅ Todos los artefactos guardados:')
print('   models/preprocessor.pkl')
print('   data/processed/{train,val,test}.csv')
print('   data/processed/{X,y}_{train,val,test}.npy')
print('   data/processed/feature_names.json')
print('\n→ Siguiente: Notebook 03 — ML Clásico + AutoML')